# Part 3 실습 — LangChain LCEL 체인 구성

## 학습 목표
- LangChain v1.x의 올바른 import 사용법을 익힌다
- LCEL(파이프 `|`) 방식으로 체인을 구성하고 실행한다
- 스트리밍, 일괄 처리, 메모리 포함 체인을 만들어본다

## 사전 설치 패키지
```
langchain langchain-openai langchain-community langchain-text-splitters python-dotenv
```

In [2]:
import os
from dotenv import load_dotenv, find_dotenv

# load_dotenv(find_dotenv(), override=True)
load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")

if api_key is None:
    raise ValueError(".env 파일에서 OPEN_API_KEY를 찾을 수 없습니다.")

print(f"API key 확인: {api_key[:10]}...")


API key 확인: sk-proj-JO...


## step 1. 기본 LCEL 체인 - 가장 단순한 형태

프롬프트 -> LLM -> 파서 순서로 연결하는 가장 기본적인 체인입니다.

In [4]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 1. 프롬프트 템플릿 정의
prompt = ChatPromptTemplate.from_template(
    "{topic}에 대해 초등학생도 이해할 수 있도록 3줄로 설명해줘."
)

# 2. LLM 초기화 
llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)

# 3. 출력 파서
parser = StrOutputParser()

# 4. LCEL로 체인 연결 (핵심: | 파이프 기호)
chain = prompt | llm | parser

# 5. 실행
result = chain.invoke({'topic': '인공지능'})
print(result)

인공지능은 컴퓨터가 사람처럼 생각하고 배우는 기술이에요. 예를 들어, 로봇이 사람의 말을 이해하거나 게임에서 똑똑하게 움직이는 것이 인공지능 덕분이에요. 이렇게 인공지능은 우리 생활을 더 편리하게 만들어줘요!


## Step 2. 스트리밍 출력

ChatGPT처럼 글자가 하나씩 나오는 스트리밍을 구현합니다.

같은 체인에 .stream()만 바꾸면 됩니다.

In [6]:
# chain은 Stemp 1에서 만든 것을 그대로 사용
print('스트리밍 출력: ')

for chunk in chain.stream({'topic': '블록체인'}):
    print(chunk, end='', flush=True)

print() 

스트리밍 출력: 
블록체인은 정보를 안전하게 저장하는 특별한 방법이에요. 여러 사람들이 함께 정보를 기록하고, 그 기록은 누구나 볼 수 있지만 쉽게 바꿀 수 없어요. 그래서 믿을 수 있는 거래나 소통을 도와주는 기술이에요!


## Step 3. 일괄 처리 (Batch)

여러 입력을 한 번에 처리합니다. 반복문보다 빠르게 처리됩니다.

In [8]:
topics = [
    {'topic': 'RAG'},
    {'topic': 'LLM'},
    {'topic': '임베딩'}
]

results = chain.batch(topics)

for topic, result in zip(topics, results):
    print(f"=== {topic['topic']} ===")
    print(result)
    print()

=== RAG ===
RAG는 "Red, Amber, Green"의 약자로, 어떤 상황이나 문제를 색깔로 표시하는 방법이에요. 빨간색은 문제가 많고 주의가 필요하다는 뜻이고, 주황색은 조심해야 하지만 괜찮다는 뜻이에요. 초록색은 모든 것이 잘 되고 있다는 의미예요!

=== LLM ===
LLM은 "대형 언어 모델"의 줄임말로, 컴퓨터가 사람처럼 글을 읽고 쓰는 능력을 갖추게 해주는 프로그램이에요. 이 모델은 많은 책과 글을 공부해서 다양한 질문에 대답하거나 이야기를 만들어낼 수 있어요. 마치 친구와 대화하듯이 컴퓨터와 소통할 수 있게 도와준답니다!

=== 임베딩 ===
임베딩은 정보를 숫자로 바꾸는 방법이에요. 예를 들어, 단어를 숫자로 표현해서 컴퓨터가 이해할 수 있게 도와줘요. 이렇게 하면 컴퓨터가 글의 의미를 더 잘 알 수 있어요!



## Step 4. 시스템 프롬프트 포함 체인

챗봇처럼 시스템 메세지(역할)를 설정한 체인입니다.

- 'from_template()' : 하나의 프롬프트만 만들 때 사용
- 'from_messages()' : 여러 역할(System, Human, AI)을 구분하여 대화 형태의 프롬프트를 만들 떄 사용 (실무에서 더 많이 사용)




In [9]:
# 시스템 + 사용자 메세지 구조
chat_prompt = ChatPromptTemplate.from_messages([
    ('system', '당신은 Python 전문가입니다. 질문에 대해 코드 예시와 함께 설명합니다. 항상 한국어로 답변합니다.'),
    ('human', '{question}')
])

expert_chain = chat_prompt | llm | StrOutputParser()

result = expert_chain.invoke({'question': 'list comprehension이 뭔가요'})
print(result)

리스트 컴프리헨션(List Comprehension)은 파이썬에서 리스트를 간결하고 효율적으로 생성하는 방법입니다. 일반적으로 반복문을 사용하여 리스트를 생성하는 대신, 한 줄의 코드로 리스트를 만들 수 있게 해줍니다. 

리스트 컴프리헨션의 기본 문법은 다음과 같습니다:

```python
[표현식 for 항목 in iterable if 조건]
```

여기서 `표현식`은 리스트의 각 요소를 어떻게 변형할지를 정의하고, `항목`은 iterable(예: 리스트, 튜플 등)에서 가져온 각 요소를 나타냅니다. `조건`은 선택적으로 사용할 수 있으며, 특정 조건을 만족하는 항목만 포함할 수 있습니다.

### 예시 1: 기본적인 리스트 컴프리헨션

1부터 10까지의 숫자의 제곱을 포함하는 리스트를 생성해보겠습니다.

```python
squares = [x**2 for x in range(1, 11)]
print(squares)
```

출력:
```
[1, 4, 9, 16, 25, 36, 49, 64, 81, 100]
```

### 예시 2: 조건을 포함한 리스트 컴프리헨션

1부터 10까지의 숫자 중에서 짝수의 제곱만 포함하는 리스트를 생성해보겠습니다.

```python
even_squares = [x**2 for x in range(1, 11) if x % 2 == 0]
print(even_squares)
```

출력:
```
[4, 16, 36, 64, 100]
```

이처럼 리스트 컴프리헨션을 사용하면 코드가 간결해지고 가독성이 높아집니다. 반복문을 사용할 때보다 더 직관적으로 리스트를 생성할 수 있습니다.


## Step 5. 멀티턴 대화 - 대화 히스토리 관리

이전 대화를 기억하는 챗봇을 만들어봅니다.

LCEL에서는 대화 히스토리를 messages 리스트로 직접 관리합니다.

In [10]:
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage

# 대화 히스토리를 저장할 리스트
chat_history = [
    SystemMessage(content = '당신은 친절한 AI 어시스턴트입니다. 한국어로 답변합니다.')
]

def chat(user_input: str) -> str:
    """사용자 메세지를 받아 응답을 반환하고 히스토리를 업데이트합니다."""
    # 사용자 메세지를 추가
    chat_history.append(HumanMessage(content=user_input))

    # LLM 호출 (전체 히스토리 전달)
    response = llm.invoke(chat_history)

    # AI 응답을 히스토리에 추가
    chat_history.append(response)

    return response.content

# 대화 테스트
print('사용자: ', '안녕하세요! 제 이름은 김지수예요.')
print('AI', chat('안녕하세요! 제 이름은 김지수예요.'))
print()
print('사용자: ', '제 이름이 뭐라고 했죠?')
print('AI', chat('제 이름이 뭐라고 했죠?')) # 이전 대화를 기억함

사용자:  안녕하세요! 제 이름은 김지수예요.
AI 안녕하세요, 김지수님! 만나서 반갑습니다. 어떻게 도와드릴까요?

사용자:  제 이름이 뭐라고 했죠?
AI 김지수님이라고 하셨습니다! 맞나요?


## Step 6. JSON 구조화 출력

LLM의 응답을 JSON 형태로 받아서 코드에서 활용하기 쉽게 만듭니다.

In [11]:
from langchain_core.output_parsers import JsonOutputParser

json_prompt = ChatPromptTemplate.from_template("""
다음 책 정보를 JSON 형식으로 정리해줘.
반드시 아래 형식을 지켜.
{{"title": "제목", "author": "저자", "genre": "장르", "summary": "한 줄 요약"}}

책 정보: {book_info}

JSON만 출력하고 다른 설명은 하지 마.
""")

json_chain = json_prompt | llm | JsonOutputParser()

result = json_chain.invoke({
    'book_info': '해리포터, 조앤 K. 롤링, 마법사 소년의 모험 이야기'
})

print(type(result))
print(result)
print(f"제목: {result['title']}")


<class 'dict'>
{'title': '해리포터', 'author': '조앤 K. 롤링', 'genre': '판타지', 'summary': '마법사 소년의 모험 이야기'}
제목: 해리포터


## Step 7. 체인 연결 (Sequential Chain)

두 체인을 이어붙여 첫 번째 출력을 두 번째 입력으로 넘깁니다.

In [17]:
from langchain_core.runnables import RunnablePassthrough

# 1단계: 주제로 블로그 제목 생성
title_prompt = ChatPromptTemplate.from_template(
    "{topic}에 관한 블로그 글 제목을 5개 제안해줘. 번호와 제목만 출력해."
)

# 2단계: 선택한 제목으로 본문c 초안 작성
draft_prompt = ChatPromptTemplate.from_template(
    "다음 제목으로 블로그 글 도입부 3문장을 작성해줘.\n\n제목 후보들:\n{titles}"
)

# 순차 체인 구성
sequential_chain = (
    {'titles': title_prompt | llm | StrOutputParser()}
    | draft_prompt
    | llm
    | StrOutputParser()
)

result = sequential_chain.invoke({'topic': 'AI와 일자리의 미래'})
print(result)

1. AI와 일자리: 미래의 직업 세계는 어떻게 변할까?  
인공지능의 발전은 우리의 일자리 환경에 혁신적인 변화를 가져오고 있습니다. 많은 사람들이 AI가 직업을 대체할 것이라는 우려를 표명하지만, 실제로는 새로운 기회도 함께 열리고 있습니다. 이 글에서는 AI가 직업 세계에 미치는 영향과 미래의 직업이 어떻게 변화할지를 탐구해보겠습니다.

2. 인공지능 시대, 일자리의 진화와 도전  
인공지능이 우리의 일상에 깊숙이 자리 잡으면서, 일자리의 형태와 요구되는 기술도 급격히 변화하고 있습니다. 이러한 변화는 많은 이들에게 도전이자 기회로 다가오고 있습니다. 이번 글에서는 인공지능 시대에 일자리가 어떻게 진화하고 있는지, 그리고 우리가 직면한 도전 과제를 살펴보겠습니다.

3. AI가 바꾸는 직업의 풍경: 기회와 위기  
AI 기술의 발전은 직업의 풍경을 근본적으로 변화시키고 있습니다. 일부 직업은 사라질 위험에 처해 있지만, 동시에 새로운 직업군이 등장하고 있습니다. 이 글에서는 AI가 가져오는 기회와 위기를 분석하고, 미래의 직업 세계에서 우리가 어떻게 준비해야 할지를 논의해보겠습니다.

4. 일자리의 미래: AI와 인간의 협업  
AI와 인간의 협업은 미래의 일자리에서 중요한 키워드로 떠오르고 있습니다. 기술이 발전함에 따라, 인간과 AI가 함께 일하는 방식이 점점 더 중요해지고 있습니다. 이번 글에서는 AI와 인간의 협업이 어떻게 일자리의 미래를 형성할지를 살펴보겠습니다.

5. 인공지능과 노동 시장: 새로운 일자리 창출의 가능성  
인공지능의 도입은 노동 시장에 큰 변화를 가져오고 있으며, 이는 새로운 일자리 창출의 기회로 이어질 수 있습니다. 많은 사람들이 AI의 발전을 위협으로 느끼지만, 실제로는 혁신적인 직업들이 등장할 가능성도 큽니다. 이 글에서는 인공지능이 노동 시장에 미치는 영향과 새로운 일자리의 가능성을 탐구해보겠습니다.


---

## 핵심 정리

| 개념 | 코드 | 설명 |
|---|---|---|
| LCEL 체인 | `prompt \| llm \| parser` | 파이프로 연결 |
| 단일 실행 | `chain.invoke({...})` | 한 번 실행 |
| 스트리밍 | `chain.stream({...})` | 실시간 출력 |
| 일괄 처리 | `chain.batch([{...}, {...}])` | 여러 입력 동시 처리 |
| 모델 교체 | `llm = ChatAnthropic(...)` | 모델 객체만 바꾸면 됨 |